In [230]:
import pandas as pd
import json
from pathlib import Path

input_json_file = Path('../source/doctors_and_clinics_raw.geojson')
output_dir = Path('../source')
output_csv_file = output_dir / 'doctors.csv'

# A list to store our flat data
flattened_data = []

print(f"Loading {input_json_file}...")
# Open and load the Overpass JSON
with open(input_json_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

print("File loaded. Processing 'elements'...")
# Iterate over each "element" in the 'elements' list
for element in data['elements']:
    
    if 'tags' in element:
        properties = element['tags'].copy() # .copy()
                
        properties['osm_id'] = element['id']
        properties['osm_type'] = element['type']
        
        if element['type'] == 'node':
            properties['longitude'] = element.get('lon')
            properties['latitude'] = element.get('lat')
            
        elif 'center' in element:
            properties['longitude'] = element['center'].get('lon')
            properties['latitude'] = element['center'].get('lat')
        
        else:
            properties['longitude'] = None
            properties['latitude'] = None
        
        # Add the assembled row to our main list
        flattened_data.append(properties)

if not flattened_data:
    print("Warning: No elements with tags were found in the file.")
else:
    print(f"Found {len(flattened_data)} elements with tags.")

# Create a DataFrame from list
df = pd.DataFrame(flattened_data)

# 9. Save the DataFrame to a CSV
try:
    df.to_csv(output_csv_file, index=False, encoding='utf-8')
    print(f"✅ File '{output_csv_file}' was successfully created!")
except Exception as e:
    print(f"❌ An error occurred while saving the CSV: {e}")

Loading ..\source\doctors_and_clinics_raw.geojson...
File loaded. Processing 'elements'...
Found 1657 elements with tags.
✅ File '..\source\doctors.csv' was successfully created!


In [231]:
df.head()

,addr:city,addr:housenumber,addr:postcode,addr:street,amenity,healthcare,healthcare:speciality,name,opening_hours,operator,...,work_accident,building:material,type,building:parts,contact:city,contact:country,contact:housenumber,contact:postcode,contact:street,contact:suburb
0,Berlin,122,12621,Münsterberger Weg,doctors,doctor,ophthalmology,Augenarzt Dr. med. Bashar Moustafa,Mo 08:00-14:00; Tu 08:00-14:00; We 13:00-18:00...,Bashar Moustafa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Berlin,5B,13353,Willdenowstraße,doctors,doctor,general,Dr. Katja Meißner,"Mo,Th 09:00-12:00,16:00-18:00; We,Fr 09:00-12:...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Berlin,22A,13593,Obstallee,doctors,centre,general;gynaecology;paediatrics;internal;psych...,Medizinisches Versorgungszentrum Heerstraße Nord,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Berlin,12a,12621,Karlstraße,doctors,doctor,NaN,NaN,"Mo 08:15-13:00,15:00-18:00;Tu 08:15-14:00;We 0...",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,doctors,doctor,NaN,Polimedica,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [232]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1657 entries, 0 to 1656
Columns: 210 entries, addr:city to contact:suburb
dtypes: float64(2), int64(1), object(207)
memory usage: 2.7+ MB


In [233]:
print(df.columns.to_list())

['addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'amenity', 'healthcare', 'healthcare:speciality', 'name', 'opening_hours', 'operator', 'website', 'osm_id', 'osm_type', 'longitude', 'latitude', 'addr:country', 'addr:suburb', 'check_date:opening_hours', 'contact:phone', 'contact:website', 'level', 'wheelchair', 'description', 'short_name', 'email', 'fax', 'health_facility:type', 'health_specialty:family_medicine', 'health_specialty:internal_medicine', 'medical_system:western', 'phone', 'toilets:wheelchair', 'check_date', 'contact:email', 'contact:fax', 'opening_hours:url', 'image', 'outdoor_seating', 'dispensing', 'office', 'comment', 'wheelchair:description', 'health_specialty:obstetrics', 'health_specialty:reproductive_medicine', 'source', 'healthcare:alternative', 'operator:type', 'internet_access', 'toilets', 'not:brand:wikidata', 'health_specialty:ophthalmology', 'min_age', 'operator:start_date', 'health_specialty:gynaecology', 'emergency', 'health_specialty:orthop

In [234]:
df['operator'].value_counts()

operator
AnthroMed Berlin-Brandenburg gGmbH                          8
MRT-Akademie                                                5
Charite Campus Virchow Klinik                               4
Dr. med. Malgorzata Kazimierczak                            3
Helios Kliniken                                             3
                                                           ..
Dr.med. Astrid Eilers-Lönnecker;Dr.med. Stephan Rackwitz    1
Andrea Jacobshagen;Stefanie Malanowski                      1
MEDICO LEOPOLDPLATZ Service GmbH                            1
Sana Gesundheitszentren Berlin-Brandenburg                  1
Priv. Doz. Dr. med. Sabine Fitzek                           1
Name: count, Length: 423, dtype: int64

In [235]:
df['osm_id'].nunique()

1657

In [236]:
df['osm_id'].head()

0    266680057
1    268915280
2    362631112
3    407094165
4    409925339
Name: osm_id, dtype: int64

In [237]:
df['osm_type'].value_counts()

osm_type
node        1591
way           61
relation       5
Name: count, dtype: int64

In [238]:
df['level'].value_counts()

level
1        77
0        48
2        39
3        23
4        20
5        10
6         3
0.5       3
1;2;3     1
0;1;2     1
1;2       1
0;1       1
Name: count, dtype: int64

In [239]:
df['description'].nunique()

99

In [240]:
df['short_name'].nunique()

4

In [241]:
df['health_facility:type'].value_counts()

health_facility:type
office           58
health_centre     6
clinic            3
doctors           1
Name: count, dtype: int64

In [242]:
df['health_specialty:family_medicine'].value_counts()

health_specialty:family_medicine
main    13
yes      3
Name: count, dtype: int64

In [243]:
df['healthcare:speciality'].value_counts()

healthcare:speciality
general                                                                                                                 289
gynaecology                                                                                                             119
paediatrics                                                                                                              93
ophthalmology                                                                                                            87
otolaryngology                                                                                                           58
                                                                                                                       ... 
orthopaedics;acupuncture;chiropractic;trauma                                                                              1
general;ophthalmology;surgery;dermatology;gynaecology;otolaryngology;paediatrics;naturopathy;orthopaedics;psyc

In [244]:
df['health_specialty:internal_medicine'].value_counts()

health_specialty:internal_medicine
yes     10
main     2
Name: count, dtype: int64

In [245]:
df['medical_system:western'].value_counts()

medical_system:western
yes    62
Name: count, dtype: int64

In [246]:
df['opening_hours:url'].value_counts() 

opening_hours:url
http://www.arztpraxis-mehdi-zadeh.de/sprechstunden.html                        1
http://www.praxis-jessen.de/#kontakt                                           1
https://www.berghafenpraxis.de/unser-team                                      1
http://www.hno-ratmann.de/sprechzeiten.html                                    1
https://praxis-zehlendorf.de/oeffnungszeiten/                                  1
https://www.kinderarzt-zimmermann.de/                                          1
https://www.kinderaerzte-im-netz.de/aerzte/berlin/kroeber/sprechzeiten.html    1
Name: count, dtype: int64

In [247]:
df['outdoor_seating'].value_counts()

outdoor_seating
no    1
Name: count, dtype: int64

In [248]:
df['dispensing'].value_counts()

dispensing
yes    1
Name: count, dtype: int64

In [249]:
df['office'].value_counts()

office
physician    35
Name: count, dtype: int64

In [250]:
df['comment'].value_counts()

comment
eingang Saltykowstraße    1
Name: count, dtype: int64

In [251]:
df['health_specialty:obstetrics'].value_counts()

health_specialty:obstetrics
main    4
yes     1
Name: count, dtype: int64

In [252]:
df['health_specialty:reproductive_medicine'].value_counts()

health_specialty:reproductive_medicine
yes    1
Name: count, dtype: int64

In [253]:
df['source'].value_counts()

source
Geoportal Berlin / Hauskoordinaten                           18
survey                                                        9
Geoportal Berlin / k5_2012_sw_sued.zip                        2
Geoportal Berlin / Hausumringe                                2
local knowledge                                               2
http://www.dr-stammeier.de/Kontakt.html                       1
93 Unter Den Eichen                                           1
Geoportal Berlin / Hauskoordinaten;survey;local knowledge     1
Dakota 20                                                     1
survey;website                                                1
Geoportal Berlin / ALKIS Berlin - Gebäude                     1
Name: count, dtype: int64

In [254]:
df['healthcare:alternative'].value_counts()

healthcare:alternative
acupuncture    1
Name: count, dtype: int64

In [255]:
df['operator:type'].value_counts()

operator:type
private     24
public       3
business     1
Name: count, dtype: int64

In [256]:
df['internet_access'].value_counts()

internet_access
wlan    6
no      5
yes     1
Name: count, dtype: int64

In [257]:
df['toilets'].value_counts()

toilets
yes    2
Name: count, dtype: int64

In [258]:
df['not:brand:wikidata'].value_counts()

not:brand:wikidata
Q1958759    1
Name: count, dtype: int64

In [259]:
df['health_specialty:ophthalmology'].value_counts()

health_specialty:ophthalmology
yes     1
main    1
Name: count, dtype: int64

In [260]:
df['min_age'].value_counts()

min_age
4    1
Name: count, dtype: int64

In [261]:
df['health_specialty:gynaecology'].value_counts()

health_specialty:gynaecology
main    4
Name: count, dtype: int64

In [262]:
df['emergency'].value_counts()

emergency
no     8
yes    2
Name: count, dtype: int64

In [263]:
df['health_specialty:orthopaedics'].value_counts()

health_specialty:orthopaedics
main    3
Name: count, dtype: int64

In [264]:
df['opening_hours:signed'].value_counts()

opening_hours:signed
no     65
yes     1
Name: count, dtype: int64

In [265]:
df['health_specialty:family_medicine'].value_counts()

health_specialty:family_medicine
main    13
yes      3
Name: count, dtype: int64

In [266]:
df['name:en'].value_counts() 

name:en
Dermatologist Helena Dröge                                                         1
AID Friedrichshain                                                                 1
Dr. med. J. Nicklas                                                                1
Gynecological Practice of Bettina Gassen                                           1
orthoteam.berlin - orthopedic medical                                              1
Spreemedizin MVZ Rehberge (General medicine)                                       1
Oncological Outpatient Department                                                  1
Spreemedizin MVZ Berlin                                                            1
Endocrinology Berlin                                                               1
OPTICUM Eye Clinic                                                                 1
Orthomed Berlin - Bartholomäus Gabrys                                              1
Dr. (TIP/Tr) Mehmet Emin Turgut                          

In [267]:
df['health_specialty:dermatology'].value_counts()

health_specialty:dermatology
partial    1
main       1
yes        1
Name: count, dtype: int64

In [268]:
df['alt_name'].value_counts()

alt_name
Ambulantes Rehazentrum                                                        1
Augenarzt                                                                     1
bonedoctor                                                                    1
Ärztehaus                                                                     1
Orthopädische und Unfallchirurgische Praxisgemeinschaft am Leipziger Platz    1
PIA                                                                           1
Zentrum für ambulante Rehabilitation Berlin                                   1
Urologische Praxis                                                            1
Ärztehaus Müllerstraße 139                                                    1
Orthopädie & Chirurgie Stadtmitte                                             1
MVZ am St. Marien-Krankenhaus Berlin                                          1
Hautarzt-Praxis Neukölln                                                      1
Praxis für Onkologie im MVZ Hav

In [269]:
df['note'].value_counts()

note
im MVZ am Bahnhof Spandau                                                                                                                                                                                                                                5
Erreichbarkeit von den Gropiuspassagen: Schild 'GesundheitsZentrum Gropiuspassagen' folgen'; 'direkt über Woolworth 1.OG und Primark EG'; Erreichbarkeit vom Kundenparkhaus P1: 'Fahrstuhl ins 2. OG'                                                    2
Allgemeinmedizin, Diab Point, Dermatologie, Diabetolog., Gynäkologie,Kardiologie; Logopädie, Neurologie, Orthopäd. Unfallchirurg, Orthopädie Schuhtechnik & Sanitätshaus, Physiotherapie, Podologie, SC Dental Labor, Schmerzpraxis,Urologie, Zahnarz    1
Fachaerztin fuer Allgemeinmedizin                                                                                                                                                                                                                 

In [270]:
df['health_specialty:venereology'].value_counts()

health_specialty:venereology
main    1
yes     1
Name: count, dtype: int64

In [271]:
df['addr:housename'].value_counts()

addr:housename
Marheineke Markthalle        1
103                          1
Nebenhaus                    1
Haus 16                      1
ÄrzteZENTRUM Ruschestraße    1
Haus 17                      1
Haus 19                      1
Haupthaus                    1
Haus 5.2                     1
Haus 5.1                     1
Haus 5.3                     1
Haus 20                      1
Werner-Otto-Haus             1
Name: count, dtype: int64

In [272]:
df['description:de'].value_counts()

description:de
Sprechzeiten sind unterschiedlich                                                      1
Kernspintomogtaphie (MRT)                                                              1
Patienten ohne Termin werden in den ersten 30 Minuten der Öffnungszeiten angenommen    1
Name: count, dtype: int64

In [273]:
df['health_specialty:paediatrics'].value_counts()

health_specialty:paediatrics
main    4
yes     2
Name: count, dtype: int64

In [274]:
df['home_visit'].value_counts()

home_visit
yes    1
Name: count, dtype: int64

In [275]:
df['fixme'].value_counts()

fixme
Name der Praxis / des Arztes?                                              2
genaue position, level                                                     2
Hier hat möglicherweise der Betreiber der gewechselt, Website ist down.    1
type                                                                       1
Name: count, dtype: int64

In [276]:
df['addr:inclusion'].value_counts()

addr:inclusion
actual    3
Name: count, dtype: int64

In [277]:
df['deaf:description:de'].value_counts()

deaf:description:de
Patient*innen werden visuell aufgerufen.    2
Name: count, dtype: int64

In [278]:
df['health_specialty:psychotherapy'].value_counts()

health_specialty:psychotherapy
yes    2
Name: count, dtype: int64

In [279]:
df['health_specialty:traditional_chinese_medicine'].value_counts()

health_specialty:traditional_chinese_medicine
partial    1
Name: count, dtype: int64

In [280]:
df['addr:place'].value_counts()

addr:place
Garbátyplatz            2
Emser Platz             1
Heinrich-Heine-Platz    1
Berlin                  1
Name: count, dtype: int64

In [281]:
df['opening_hours:dr_weinhold'].value_counts()

opening_hours:dr_weinhold
Mo,We 08:00-12:00, 13:30-16:00; Tu 08:00-12:00; Th 08:00-12:00, 14:00-17:00; Fr 08:00-12:00    1
Name: count, dtype: int64

In [282]:
df['health_specialty:sports_medicine'].value_counts()

health_specialty:sports_medicine
additional    3
Name: count, dtype: int64

In [283]:
df['opening_hours:note'].value_counts()

opening_hours:note
Wednesday: only for private insurance    1
Name: count, dtype: int64

In [284]:
df['note:de'].value_counts()

note:de
Privatpraxis                                   1
Terminvereinbarungen Mo, Tu, Fr 12:30-13:00    1
Name: count, dtype: int64

In [285]:
df['name:de'].value_counts()

name:de
Arztpraxis Driesener Straße                                                 1
HNO-Praxis                                                                  1
AID Friedrichshain                                                          1
Hautarztpraxis Dr. Hasert                                                   1
Dr. med. Katja Rebell                                                       1
Spreemedizin MVZ Berlin                                                     1
Orthomed Berlin – Bartholomäus Gabrys - Orthopädie Friedrichshain           1
Praxis für Kinder- und Jugendpsychatrie MVZ Greven, Treuter und Kollegen    1
Name: count, dtype: int64

In [286]:
df['opening_hours:note:de'].value_counts()

opening_hours:note:de
Fr Termine nach Vereinbarung    1
Name: count, dtype: int64

In [287]:
df['start_date'].value_counts()

start_date
2003-09-03    1
1993          1
2014-10       1
2020-10       1
2023-03-01    1
2005          1
1921..1923    1
2013          1
Name: count, dtype: int64

In [288]:
df['language:tr'].value_counts()

language:tr
yes    3
Name: count, dtype: int64

In [289]:
df['opening_hours:office'].value_counts()

opening_hours:office
Mo-Th 09:00-12:00,15:00-18:00; We 10:00-12:00; Fr 09:00-12:00    1
Name: count, dtype: int64

In [290]:
df['survey:date'].value_counts()

survey:date
2022-03-02    3
Name: count, dtype: int64

In [291]:
df['building'].value_counts()

building
yes           34
commercial    11
civic          4
healthcare     3
office         2
clinic         2
house          1
apartments     1
Name: count, dtype: int64

In [292]:
df['ele'].value_counts()

ele
20    3
Name: count, dtype: int64

In [293]:
df['type'].value_counts()

type
multipolygon    5
Name: count, dtype: int64

In [294]:
df['check_date:opening_hours'].value_counts()


check_date:opening_hours
2022-11-27    5
2025-04-24    5
2024-08-10    4
2025-10-15    4
2025-10-16    4
             ..
2023-03-31    1
2024-10-16    1
2022-07-19    1
2025-05-05    1
2023-10-10    1
Name: count, Length: 216, dtype: int64

In [301]:
columns_to_drop=['osm_type','operator','level', 'short_name','health_facility:type', 'health_specialty:internal_medicine', 'medical_system:western', 'check_date', 'image', 
                 'outdoor_seating', 'dispensing', 'office', 'comment','health_specialty:obstetrics','health_specialty:reproductive_medicine', 'source','healthcare:alternative',
                 'operator:type','internet_access', 'toilets','not:brand:wikidata', 'health_specialty:ophthalmology', 'min_age', 'operator:start_date', 'health_specialty:gynaecology', 'emergency',
                 'health_specialty:orthopaedics','opening_hours:signed', 'health_specialty:dermatology','contact:fax', 'fax', 'health_specialty:family_medicine','name:en', 
                 'health_specialty:surgery', 'alt_name', 'health_specialty:venereology', 'addr:housename', 'health_specialty:paediatrics','home_visit', 'addr:inclusion', 
                 'blind:description:de', 'deaf:description:de', 'health_specialty:psychosomatic_medicine', 'health_specialty:psychotherapy',
                 'health_specialty:traditional_chinese_medicine', 'addr:place', 'payment:cash', 'payment:credit_cards', 'payment:debit_cards', 'contact:facebook', 'name:ru', 
                 'opening_hours:reception', 'operator:wikidata', 'health_specialty:emergency_medicine', 'health_specialty:sports_medicine', 'payment:mastercard', 'payment:visa',
                 'layer', 'opening_hours:note', 'note:de', 'health_specialty:diagnostic_radiology', 'health_specialty:dentistry', 'health_specialty:maxillofacial_surgery', 
                 'access', 'doctor', 'instagram', 'health_specialty:allergology', 'health_specialty:pulmonology', 'disease:hiv', 'disease:stds', 'start_date', 'blind:description', 
                 'operator:wikipedia', 'health_specialty:gastroenterology', 'internet_access:fee', 'internet_access:ssid', 'addr:floor', 'elevator',  'language:en', 'language:es',
                 'language:fr', 'language:tr', 'wheelchair:description:de', 'air_conditioning', 'health_service:prevention', 'health_service:test', 'lgbtq:trans',  
                 'health_specialty:cardiology', 'healthcare:speciality:de', 'health_specialty:paediatric_gastroenterology', 'beauty', 'opening_hours:covid19', 'name:ja', 'branch',
                 'name:signed', 'old_name', 'payment:american_express', 'payment:coins', 'payment:maestro', 'payment:notes', 'payment:telephone_cards', 'capacity', 'day_surgery', 
                 'health_specialty:pain_medicine', 'health_specialty:andrology', 'health_specialty:urology', 'health_specialty:optometry', 'survey:date', 'counselling_type:nutrition', 
                 'health_service:counselling', 'health_specialty:acupuncture', 'health_specialty:homoeopathy', 'health_specialty:mind_body_intervention','health_specialty:naturopathy', 
                 'health_specialty:osteopathy', 'health_specialty:phythotherapy', 'insurance:health', 'health_specialty:adult_psychiatry', 'health_specialty:neurology', 'reservation', 
                 'noname', 'payment:privat', 'contact:instagram', 'health_specialty:neonatology', 'health_specialty:ear_nose_throat', 'social_facility:for',  'language:de', 
                 'language:ku', 'language:ru', 'language:ar', 'language:th', 'fixme:type', 'health_service:examination', 'smoking', 'building', 'building:levels', 'neighborhood', 
                 'roof:shape', 'height', 'roof:levels', 'roof:colour', 'social_facility', 'wikidata', 'wikimedia_commons', 'deaf:description', 'official_name', 'brand', 'brand:wikidata', 
                 'wikipedia', 'roof:height', 'heritage', 'heritage:operator', 'lda:criteria', 'ref:lda', 'building:colour', 'ele', 'health_specialty:anesthesiology', 
                 'health_specialty:hand_surgery', 'health_specialty:trauma_surgery', 'not:operator:wikidata', 'work_accident', 'building:material', 'type', 'building:parts', 
                  'contact:city', 'contact:country', 'contact:housenumber', 'contact:postcode', 'contact:street', 'contact:suburb','check_date:opening_hours']
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1657 entries, 0 to 1656
Data columns (total 31 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   addr:city                  1146 non-null   object 
 1   addr:housenumber           1269 non-null   object 
 2   addr:postcode              1210 non-null   object 
 3   addr:street                1272 non-null   object 
 4   amenity                    1657 non-null   object 
 5   healthcare                 1647 non-null   object 
 6   healthcare:speciality      1437 non-null   object 
 7   name                       1611 non-null   object 
 8   opening_hours              1129 non-null   object 
 9   website                    910 non-null    object 
 10  osm_id                     1657 non-null   int64  
 11  longitude                  1591 non-null   float64
 12  latitude                   1591 non-null   float64
 13  addr:country               755 non-null    objec

In [296]:
   
all_filled = df[['website', 'contact:website','heritage:website','url']].notnull().all(axis=1)

print(df[all_filled][['website', 'contact:website','heritage:website','url']])

Empty DataFrame
Columns: [website, contact:website, heritage:website, url]
Index: []


In [297]:
df['website'] = df['website'].fillna(df['contact:website'])
df['website'] = df['website'].fillna(df['heritage:website'])
df['website'] = df['website'].fillna(df['url'])
df.drop(columns=['contact:website','heritage:website','url'], inplace=True, errors='ignore')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1657 entries, 0 to 1656
Data columns (total 37 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   addr:city                  1146 non-null   object 
 1   addr:housenumber           1269 non-null   object 
 2   addr:postcode              1210 non-null   object 
 3   addr:street                1272 non-null   object 
 4   amenity                    1657 non-null   object 
 5   healthcare                 1647 non-null   object 
 6   healthcare:speciality      1437 non-null   object 
 7   name                       1611 non-null   object 
 8   opening_hours              1129 non-null   object 
 9   website                    910 non-null    object 
 10  osm_id                     1657 non-null   int64  
 11  longitude                  1591 non-null   float64
 12  latitude                   1591 non-null   float64
 13  addr:country               755 non-null    objec

In [298]:
all_filled = df[['contact:phone', 'phone','phone_1','contact:mobile','mobile']].notnull().all(axis=1)

print(df[all_filled][['contact:phone', 'phone','phone_1','contact:mobile','mobile']])

Empty DataFrame
Columns: [contact:phone, phone, phone_1, contact:mobile, mobile]
Index: []


In [299]:
df['phone'] = df['phone'].fillna(df['contact:phone'])
df['phone'] = df['phone'].fillna(df['phone'])
df['phone'] = df['phone'].fillna(df['phone_1'])
df['phone'] = df['phone'].fillna(df['contact:mobile'])
df['phone'] = df['phone'].fillna(df['mobile'])
df.drop(columns=['contact:phone','phone','phone_1','contact:mobile','mobile'], inplace=True, errors='ignore')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1657 entries, 0 to 1656
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   addr:city                  1146 non-null   object 
 1   addr:housenumber           1269 non-null   object 
 2   addr:postcode              1210 non-null   object 
 3   addr:street                1272 non-null   object 
 4   amenity                    1657 non-null   object 
 5   healthcare                 1647 non-null   object 
 6   healthcare:speciality      1437 non-null   object 
 7   name                       1611 non-null   object 
 8   opening_hours              1129 non-null   object 
 9   website                    910 non-null    object 
 10  osm_id                     1657 non-null   int64  
 11  longitude                  1591 non-null   float64
 12  latitude                   1591 non-null   float64
 13  addr:country               755 non-null    objec

In [ ]:
all_filled = df[['email', 'contact:email']].notnull().all(axis=1)

print(df[all_filled][['email', 'contact:email']])

                              email                  contact:email
1331  karlshorst@ihre-hno-aerzte.de  karlshorst@ihre-hno-aerzte.de
1622    praxis@mvz-kinderexperts.de    praxis@mvz-kinderexperts.de


In [306]:
df.drop(columns=['contact:email'], inplace=True, errors='ignore')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1657 entries, 0 to 1656
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   addr:city                  1146 non-null   object 
 1   addr:housenumber           1269 non-null   object 
 2   addr:postcode              1210 non-null   object 
 3   addr:street                1272 non-null   object 
 4   amenity                    1657 non-null   object 
 5   healthcare                 1647 non-null   object 
 6   healthcare:speciality      1437 non-null   object 
 7   name                       1611 non-null   object 
 8   opening_hours              1129 non-null   object 
 9   website                    910 non-null    object 
 10  osm_id                     1657 non-null   int64  
 11  longitude                  1591 non-null   float64
 12  latitude                   1591 non-null   float64
 13  addr:country               755 non-null    objec

In [307]:
all_filled = df[['website', 'opening_hours:url']].notnull().all(axis=1)

print(df[all_filled][['website', 'opening_hours:url']])

                                                website  \
6                 http://www.arztpraxis-mehdi-zadeh.de/   
227                       https://www.praxis-jessen.de/   
618                     https://www.berghafenpraxis.de/   
620                           http://www.hno-ratmann.de   
835                       https://praxis-zehlendorf.de/   
1029              https://www.kinderarzt-zimmermann.de/   
1421  https://www.kinderaerzte-im-netz.de/aerzte/ber...   

                                      opening_hours:url  
6     http://www.arztpraxis-mehdi-zadeh.de/sprechstu...  
227                http://www.praxis-jessen.de/#kontakt  
618           https://www.berghafenpraxis.de/unser-team  
620         http://www.hno-ratmann.de/sprechzeiten.html  
835       https://praxis-zehlendorf.de/oeffnungszeiten/  
1029              https://www.kinderarzt-zimmermann.de/  
1421  https://www.kinderaerzte-im-netz.de/aerzte/ber...  


In [308]:
df.drop(columns=['opening_hours:url'], inplace=True, errors='ignore')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1657 entries, 0 to 1656
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   addr:city                  1146 non-null   object 
 1   addr:housenumber           1269 non-null   object 
 2   addr:postcode              1210 non-null   object 
 3   addr:street                1272 non-null   object 
 4   amenity                    1657 non-null   object 
 5   healthcare                 1647 non-null   object 
 6   healthcare:speciality      1437 non-null   object 
 7   name                       1611 non-null   object 
 8   opening_hours              1129 non-null   object 
 9   website                    910 non-null    object 
 10  osm_id                     1657 non-null   int64  
 11  longitude                  1591 non-null   float64
 12  latitude                   1591 non-null   float64
 13  addr:country               755 non-null    objec

In [309]:
all_filled = df[['name', 'name:de']].notnull().all(axis=1)

print(df[all_filled][['name', 'name:de']])

                                                   name  \
333                         Arztpraxis Driesener Straße   
346                                          HNO-Praxis   
350                                  AID Friedrichshain   
380                           Hautarztpraxis Dr. Hasert   
1139                              Dr. med. Katja Rebell   
1269                            Spreemedizin Berlin MVZ   
1394                                    Orthomed Berlin   
1574  Praxis für Kinder- und Jugendpsychatrie MVZ Gr...   

                                                name:de  
333                         Arztpraxis Driesener Straße  
346                                          HNO-Praxis  
350                                  AID Friedrichshain  
380                           Hautarztpraxis Dr. Hasert  
1139                              Dr. med. Katja Rebell  
1269                            Spreemedizin MVZ Berlin  
1394  Orthomed Berlin – Bartholomäus Gabrys - Orthop...  
1574

In [310]:
df.drop(columns=['name:de'], inplace=True, errors='ignore')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1657 entries, 0 to 1656
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   addr:city                  1146 non-null   object 
 1   addr:housenumber           1269 non-null   object 
 2   addr:postcode              1210 non-null   object 
 3   addr:street                1272 non-null   object 
 4   amenity                    1657 non-null   object 
 5   healthcare                 1647 non-null   object 
 6   healthcare:speciality      1437 non-null   object 
 7   name                       1611 non-null   object 
 8   opening_hours              1129 non-null   object 
 9   website                    910 non-null    object 
 10  osm_id                     1657 non-null   int64  
 11  longitude                  1591 non-null   float64
 12  latitude                   1591 non-null   float64
 13  addr:country               755 non-null    objec

In [312]:
df['addr:country'].unique()

array([nan, 'DE'], dtype=object)

In [313]:
df['addr:city'].unique()

array(['Berlin', nan], dtype=object)